# What did the fly contribute to ARC-AGI-3?

I compiled a graph of 166,700 neurons from the adult male fly's central nervous system. I assigned numerical weights from anatomical synaptic contact counts, then compared active circuits with copies whose neurons kept the same numbers of connections but changed partners.

The three 0.15 Kaggle submissions used a static map of the fly's eye. All three reported `neuronal_dynamics_active=false`. The 24,994,595-edge deployment bundle supplied anatomical metadata. Local experiments tested the dynamics separately. This notebook plots the recorded scores and optionally recalculates a structural measurement from the original graph files. [Public repository](https://github.com/msafdari/FlyGI) · [Kaggle paper](https://www.kaggle.com/competitions/arc-prize-2026-paper-track/writeups/flygi-connectome-attribution-audit)

## What changed when I rewired the circuits?

| Experiment | Calibration | Measured result |
| --- | --- | --- |
| Mushroom body, one-shot visual association | Held-out active fraction 0.0741, inside the registered 0.03 to 0.15 range. Projection neurons supplied 62% to 87% of measured drive to Kenyon cells. | On five held-out ARC games across three seeds, accuracy was **60%** with biological wiring and **80%** with two connection blocks rewired. Permuting which cells received credit reduced accuracy to 40%. |
| Motion direction, T4/T5 | A 2,701-cell visual patch used parameters trained by flyvis. Its median direction-selectivity index was 0.3216 against a 0.30 threshold. | A trained readout of 498 scored cells decoded direction at **38.75%** versus **43.06%** with rewiring. Each of three splits held out four games. The shift matcher scored 100% on the events it helped select. |
| Heading retention, compass | Earlier neuron-level simulations either lost localized activity or drifted toward a preferred heading. | Off ARC, I grouped 46 EPG and 42 Delta7 neurons into 16 units, symmetrized their counts and fitted the couplings using the Biswas et al. recipe. Two of three settings held all eight initialized headings in darkness. The same search found **zero** valid fits across three rewires. My fitted values differed from the preprint. |
| Object position, LC cells | The rate model tested 21 gain and leak settings. | Even the best rank correlation with position was negative at −0.203 against a +0.50 target. The circuit stopped at the biology check. |

The eight compass initializations rotated the same heading pattern in 45° increments. The experiment measured retention after the model was given the heading. [ARC circuit records](https://github.com/msafdari/FlyGI/tree/main/results/manifests/verdict_program_20260930) · [Compass model](https://github.com/msafdari/FlyGI/blob/main/scripts/arenas/cx_biswas_killtest.py)

## Three submissions, one hidden score

The first agent supplied the baseline, the second recognized two public-game strategies, and the third recognized thirteen. In a 35-minute local competition run, the third agent completed **73 levels and four whole games**. Three identical leaderboard readings establish the change in displayed score. Hidden per-game strategy activations were unobserved. [Gateway records](https://github.com/msafdari/FlyGI/tree/main/results/manifests)

In [ ]:
import matplotlib.pyplot as plt

# These are transcribed measurements from the local gateways and Kaggle.
local = [0.551859, 3.044167, 28.742260]
hidden = [0.15, 0.15, 0.15]
labels = ["Baseline\n56620463", "Two grammars\n56650224", "Thirteen grammars\n56683298"]
print(f"Local score increased {local[-1] / local[0]:.1f} times.")
fig, ax = plt.subplots(figsize=(9, 4.5))
for offset, scores, name, color in [(-0.16, local, "35-minute local gateway", "#2363a5"),
                                     (0.16, hidden, "Kaggle hidden score", "#ae4e1e")]:
    bars = ax.bar([i + offset for i in range(3)], scores, width=0.30, label=name, color=color)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in scores], padding=3)
ax.set_xticks(range(3), labels)
ax.set_ylim(0, 32)
ax.set_ylabel("Score")
ax.set_title("A 52-fold local gain. Three hidden scores of 0.15.")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Where the eye map earned its place

On public game `r11l`, the click controller completed level one in **6 and 15 actions** with anatomical retinal coordinates, against **199 and 255** with permuted coordinates on the same two seeds. On the remaining seed the real map finished in 57 actions and the permuted map never finished. Across all 14 game/seed pairs where *both* controllers completed level one, the median ratio of permuted to anatomical action cost was **1.06**. One-sided completions fall outside that ratio. Kaggle received only the anatomical version.

A separate class-1 ARC test could not reliably separate a long-memory reservoir from its controls on score. That limits how much the tested interface can tell us about the simulated fly. [League](https://github.com/msafdari/FlyGI/blob/main/LEAGUE.md) · [Research synthesis](https://github.com/msafdari/FlyGI/blob/main/docs/arenas/SYNTHESIS.md)

## Replay a structural measurement

On the original heuristic-signed graph with 24,559,055 edges, the power-iteration estimate was **0.8289**. Three degree-preserving rewires gave **0.2360, 0.2348 and 0.2219**. The real estimate was **3.59 times** their mean. This measurement uses the signed weight matrix itself. The modulator-preserving Kaggle bundle is a different graph. [Recorded comparison](https://github.com/msafdari/FlyGI/blob/main/results/manifests/degree_preserving_comparison_20260921.json)

The next cell looks for the historical signed build in the repository's default data directory. For a hosted notebook, attach the historical build and set `FLYGI_SIGNED_RUNTIME` to the directory containing its `graph.npz`. Put any original rewires in their `nulls/degree_preserving/seed-YYYYMMDD` subdirectories. The cell verifies each archive's SHA-256 before reconstructing the matrix with rows for receiving neurons and columns for sending neurons. It repeats the original seeded power iteration and checks the four decimal places reported. [Estimator used in the experiment](https://github.com/msafdari/FlyGI/blob/b6cd353f23d771d599bd871b73e7a4683458788d/src/flygi/connectome/normalize.py)

In [ ]:
import hashlib
import os
from pathlib import Path

import numpy as np
from scipy.sparse import csr_matrix

# File digests identify the four original archives independently of their paths.
reference = {
    ".": ("45e332d90ea4e881ece5d4d72216955bd53eec922aa60955a69db510272eff61", 0.8289),
    "nulls/degree_preserving/seed-20260921": ("8cb83109b442fe007c4a15c52f7d16febdb66b5871fc0868b20f4cbf3af16a6f", 0.2360),
    "nulls/degree_preserving/seed-20260922": ("138c7ee98abc29a1ef3f5115a743e32761ff28b3e104a89ae030c5680b092b14", 0.2348),
    "nulls/degree_preserving/seed-20260923": ("53f00664527dc73498d68e0394d7c616f022e9f3f48e273d4ef740ad48083776", 0.2219),
}


def historical_estimate(path, digest):
    with path.open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != digest:
            raise ValueError(f"{path}: SHA-256 differs from the historical archive")
    with np.load(path, allow_pickle=False) as arrays:
        n = len(arrays["node_ids"])
        matrix = csr_matrix((arrays["weights"], arrays["indices"], arrays["indptr"]),
                            shape=(n, n), dtype=np.float64)
    vector = np.random.default_rng(0).standard_normal(n)
    vector /= np.linalg.norm(vector)
    previous = 0.0
    for _ in range(300):
        product = matrix @ vector
        estimate = float(np.linalg.norm(product))
        if not np.isfinite(estimate) or estimate == 0:
            raise ArithmeticError(f"{path}: unexpected power-iteration norm {estimate}")
        vector = product / estimate
        if abs(estimate - previous) <= 1e-6 * max(1.0, estimate):
            break
        previous = estimate
    return estimate


configured = os.environ.get("FLYGI_SIGNED_RUNTIME")
runtime = Path(configured or "data/processed/male-cns-v1.0/literal-rate-signed")
measured = {}
if not configured and not (runtime / "graph.npz").exists():
    print("Records only. To recompute, set FLYGI_SIGNED_RUNTIME to the historical signed build.")
else:
    for label, (digest, published) in reference.items():
        path = runtime / label / "graph.npz"
        if label != "." and not path.is_file():
            print(f"{label}: original rewire unavailable")
            continue
        value = historical_estimate(path, digest)
        if round(value, 4) != published:
            raise AssertionError(f"{label}: computed {value:.6f}, expected {published:.4f}")
        measured[label] = value
        print(f"{label}: computed {value:.6f} from verified archive")
    if len(measured) == len(reference):
        rewires = [value for label, value in measured.items() if label != "."]
        print(f"Real / mean of three rewires: {measured['.'] / np.mean(rewires):.2f}x")

## Data and provenance

The MaleCNS data are distributed under CC BY 4.0. The source study describes the male fly's central nervous system, including its brain and ventral nerve cord. The FlyGI runtime arrays are modified derivatives. [Dataset](https://male-cns.janelia.org/) · [Source study](https://doi.org/10.1016/j.cell.2026.08.015)

The worklog records protocols and submission identifiers. For a local graph build, follow the repository quick start with `configs/literal_rate_signed.yaml` for the historical signed experiment. [Worklog](https://github.com/msafdari/FlyGI/blob/main/WORKLOG.md) · [Repository](https://github.com/msafdari/FlyGI) · [Experiment registry](https://github.com/msafdari/FlyGI/blob/main/EXPERIMENTS.yaml)